# Facebook Friendship Network with BigQuery and Colab

This notebook was generated from the provided article text. It is organized with Markdown explanation cells and runnable Python code cells.

**Before running:** replace `PROJECT_ID = "your-project-id"` with your Google Cloud project ID. You also need BigQuery enabled in that project.


## Real Dataset Example: Facebook Friendship Network with BigQuery and Colab

So far, we used a small made-up neighborhood friendship map.

Now let’s use a real online network dataset.

We will use the Stanford SNAP Facebook Social Circles dataset. It contains anonymized Facebook friendship connections. Each user is represented by a number, and each friendship is represented as a connection between two user IDs. The dataset has 4,039 users and 88,234 friendship edges.

This is a good beginner dataset because it matches our analogy very well:

```text
Node = Facebook user
Edge = friendship connection
Graph = the full friendship network
```

We will use Google Colab to download the dataset, then load it into BigQuery, create a BigQuery property graph, analyze it, visualize it, and prepare it for link prediction.

---

## 1. Install Packages in Colab


In [ ]:
# Install the libraries we need.
# google-cloud-bigquery connects Colab to BigQuery.
# pandas helps us work with tables.
# networkx helps us analyze and visualize graph data.
# matplotlib helps us draw the network.

!pip install --upgrade google-cloud-bigquery pandas-gbq networkx matplotlib scikit-learn


---

## 2. Authenticate Colab with Google Cloud


In [ ]:
# Authenticate this Colab notebook with your Google account.

from google.colab import auth
auth.authenticate_user()

print("Authenticated.")


---

## 3. Connect to BigQuery

Replace `your-project-id` with your real Google Cloud project ID.


In [ ]:
# Import the BigQuery client.
from google.cloud import bigquery

# Replace this with your Google Cloud project ID.
PROJECT_ID = "your-project-id"

# Create the BigQuery client.
client = bigquery.Client(project=PROJECT_ID)

print("Connected to BigQuery.")


BigQuery’s Python client can load pandas DataFrames into BigQuery tables, which is useful here because we will download the graph file into Colab first and then upload it to BigQuery.

---

## 4. Download the Real Facebook Network Dataset


In [ ]:
# Import pandas for reading the edge list.
import pandas as pd

# URL for the SNAP Facebook combined edge list.
url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"

# Read the compressed text file directly from the web.
# The file has two columns: source user and target user.
edges_df = pd.read_csv(
    url,
    sep=" ",
    header=None,
    names=["source_id", "target_id"]
)

# Show the first few friendship connections.
edges_df.head()


Each row means two anonymized Facebook users are connected.

For example:

```text
source_id | target_id
----------|----------
0         | 1
0         | 2
0         | 3
```

That means user `0` is connected to users `1`, `2`, and `3`.

---

## 5. Create a Nodes Table

The edge file gives us friendships, but BigQuery Graph also needs a node table.

We can create the node table by collecting every unique user ID that appears in the edge list.


In [ ]:
# Get all unique user IDs from both columns.
all_users = pd.concat([
    edges_df["source_id"],
    edges_df["target_id"]
]).drop_duplicates()

# Create a node table.
users_df = pd.DataFrame({
    "user_id": all_users
})

# Add a simple display name.
# Since the data is anonymized, we will call them User 0, User 1, etc.
users_df["user_name"] = "User " + users_df["user_id"].astype(str)

# Sort users by ID.
users_df = users_df.sort_values("user_id").reset_index(drop=True)

# Show the first few users.
users_df.head()


Now we have:

```text
Users table = nodes
Friendships table = edges
```

---

## 6. Add an Edge ID and Simple Properties

BigQuery property graphs work better when edge rows have a unique key.

Let’s add an `edge_id`.


In [ ]:
# Add a unique edge ID.
edges_df = edges_df.reset_index().rename(columns={"index": "edge_id"})

# Add a simple weight.
# This dataset does not include friendship strength,
# so we set every friendship weight to 1.
edges_df["weight"] = 1

# Show the first few edges.
edges_df.head()


The edge table now looks like this:

```text
edge_id | source_id | target_id | weight
--------|-----------|-----------|-------
0       | 0         | 1         | 1
1       | 0         | 2         | 1
2       | 0         | 3         | 1
```

---

## 7. Create a BigQuery Dataset


In [ ]:
# Create a BigQuery dataset for this graph project.

dataset_sql = f"""
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.facebook_graph_demo`
OPTIONS(location = "US");
"""

client.query(dataset_sql).result()

print("Dataset created.")


---

## 8. Load the DataFrames into BigQuery


In [ ]:
# Define BigQuery table names.
users_table = f"{PROJECT_ID}.facebook_graph_demo.Users"
edges_table = f"{PROJECT_ID}.facebook_graph_demo.Friendships"

# Load the users DataFrame into BigQuery.
job_users = client.load_table_from_dataframe(
    users_df,
    users_table
)
job_users.result()

# Load the edges DataFrame into BigQuery.
job_edges = client.load_table_from_dataframe(
    edges_df,
    edges_table
)
job_edges.result()

print("Users and Friendships tables loaded into BigQuery.")


Check the row counts:


In [ ]:
count_sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{users_table}`) AS number_of_users,
  (SELECT COUNT(*) FROM `{edges_table}`) AS number_of_friendships;
"""

client.query(count_sql).to_dataframe()


You should see around:

```text
number_of_users | number_of_friendships
----------------|----------------------
4039            | 88234
```

---

## 9. Create a BigQuery Property Graph

BigQuery Graph lets you create a graph from existing BigQuery tables using `CREATE PROPERTY GRAPH`. The graph is a logical view over the node and edge tables, meaning the data stays in the underlying tables.


In [ ]:
# Create a BigQuery property graph.
# Users are nodes.
# Friendships are edges.

create_graph_sql = f"""
CREATE OR REPLACE PROPERTY GRAPH `{PROJECT_ID}.facebook_graph_demo.FacebookGraph`
  NODE TABLES (
    `{users_table}` AS Users
      KEY (user_id)
      LABEL User
      PROPERTIES (user_id, user_name)
  )
  EDGE TABLES (
    `{edges_table}` AS Friendships
      KEY (edge_id)
      SOURCE KEY (source_id) REFERENCES Users (user_id)
      DESTINATION KEY (target_id) REFERENCES Users (user_id)
      LABEL Friend
      PROPERTIES (edge_id, source_id, target_id, weight)
  );
"""

client.query(create_graph_sql).result()

print("BigQuery property graph created.")


One important note: Facebook friendships are naturally undirected, meaning if user A is friends with user B, user B is also friends with user A. BigQuery property graph edges are modeled with a source and destination, so undirected relationships can be modeled as two directed edges or queried with undirected edge patterns. Google’s graph schema documentation also mentions that reciprocal relationships like friendships can be modeled using two directed edges, one in each direction.

---

## 10. Query Basic Graph Information

Let’s count users and friendships from BigQuery.


In [ ]:
basic_sql = f"""
SELECT
  COUNT(DISTINCT user_id) AS number_of_users
FROM `{users_table}`;
"""

client.query(basic_sql).to_dataframe()


In [ ]:
edge_count_sql = f"""
SELECT
  COUNT(*) AS number_of_friendship_edges
FROM `{edges_table}`;
"""

client.query(edge_count_sql).to_dataframe()


---

## 11. Find the Most Connected Users

The number of friends a user has is called degree.

Because the friendship is undirected, we count connections in both directions.


In [ ]:
degree_sql = f"""
WITH all_friend_directions AS (
  SELECT source_id AS user_id, target_id AS friend_id
  FROM `{edges_table}`

  UNION ALL

  SELECT target_id AS user_id, source_id AS friend_id
  FROM `{edges_table}`
)

SELECT
  u.user_id,
  u.user_name,
  COUNT(DISTINCT afd.friend_id) AS degree
FROM all_friend_directions afd
JOIN `{users_table}` u
  ON afd.user_id = u.user_id
GROUP BY u.user_id, u.user_name
ORDER BY degree DESC
LIMIT 20;
"""

top_users_df = client.query(degree_sql).to_dataframe()

top_users_df


Plain-language meaning:

These are the most connected users in the Facebook friendship network.

They are the people with the largest number of direct friendship connections.

---

## 12. Visualize a Small Subgraph in Colab

The full network has more than 4,000 nodes and 88,000 edges, so drawing everything can look messy.

Instead, let’s visualize the local network around one highly connected user.


In [ ]:
# Choose one user to visualize.
# We will use the most connected user from the previous query.
center_user = int(top_users_df.iloc[0]["user_id"])

center_user


Now pull this user’s direct friends.


In [ ]:
# Get the center user and their direct friends.

subgraph_sql = f"""
WITH neighbors AS (
  SELECT source_id AS user_id, target_id AS friend_id
  FROM `{edges_table}`
  WHERE source_id = {center_user}

  UNION ALL

  SELECT target_id AS user_id, source_id AS friend_id
  FROM `{edges_table}`
  WHERE target_id = {center_user}
)

SELECT
  user_id AS source,
  friend_id AS target
FROM neighbors
LIMIT 100;
"""

sub_edges_df = client.query(subgraph_sql).to_dataframe()

sub_edges_df.head()


Now draw the subgraph.


In [ ]:
# Import graph libraries.
import networkx as nx
import matplotlib.pyplot as plt

# Create a graph from the subgraph edge table.
G_sub = nx.from_pandas_edgelist(
    sub_edges_df,
    source="source",
    target="target",
    create_using=nx.Graph()
)

# Create layout positions.
pos = nx.spring_layout(G_sub, seed=42)

# Draw the network.
plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(G_sub, pos, node_size=300)
nx.draw_networkx_edges(G_sub, pos, width=1)
nx.draw_networkx_labels(G_sub, pos, font_size=7)

plt.title(f"Local Friendship Network Around User {center_user}")
plt.axis("off")
plt.show()


---

## Figure: Local Friendship Network

This figure shows one user and a sample of their direct friends.

```text
          friend
            |
friend -- center user -- friend
            |
          friend
```

In the real visualization, you will see many users connected to the same center node.

This helps readers understand degree visually.

---

## 13. Calculate Degree Centrality in BigQuery

Degree centrality is the degree divided by the maximum possible number of friends.

If there are 4,039 users, each user could theoretically connect to 4,038 other users.


In [ ]:
degree_centrality_sql = f"""
WITH all_friend_directions AS (
  SELECT source_id AS user_id, target_id AS friend_id
  FROM `{edges_table}`

  UNION ALL

  SELECT target_id AS user_id, source_id AS friend_id
  FROM `{edges_table}`
),

degree_table AS (
  SELECT
    user_id,
    COUNT(DISTINCT friend_id) AS degree
  FROM all_friend_directions
  GROUP BY user_id
),

user_count AS (
  SELECT COUNT(*) AS n
  FROM `{users_table}`
)

SELECT
  d.user_id,
  d.degree,
  ROUND(d.degree / (uc.n - 1), 4) AS degree_centrality
FROM degree_table d
CROSS JOIN user_count uc
ORDER BY degree_centrality DESC
LIMIT 20;
"""

degree_centrality_df = client.query(degree_centrality_sql).to_dataframe()

degree_centrality_df


Plain-language meaning:

A user with high degree centrality is directly connected to a larger share of the network.

---

## 14. Shortest Path in Colab

Shortest path means:

How many friendship steps connect one user to another?

For example, user A may not directly know user B, but they may be connected through a chain of friends.

For this part, it is easier to use NetworkX in Colab.


In [ ]:
# Build a NetworkX graph from the full edge list.
G = nx.from_pandas_edgelist(
    edges_df,
    source="source_id",
    target="target_id",
    create_using=nx.Graph()
)

# Pick two users.
user_a = 0
user_b = 348

# Find the shortest path.
path = nx.shortest_path(G, source=user_a, target=user_b)

print("Shortest path:", path)
print("Number of steps:", len(path) - 1)


Plain-language meaning:

If the path is:

```text
0 -> 56 -> 348
```

Then user `0` is connected to user `348` through one mutual person, user `56`.

---

## 15. Clustering: Do Friends Know Each Other?

Clustering tells us whether a user’s friends are also friends with one another.

A high clustering score means the user belongs to a tight social circle.


In [ ]:
# Calculate clustering coefficient for each user.
clustering = nx.clustering(G)

# Convert the result to a DataFrame.
clustering_df = pd.DataFrame(
    list(clustering.items()),
    columns=["user_id", "clustering_score"]
)

# Sort from highest to lowest.
clustering_df = clustering_df.sort_values(
    "clustering_score",
    ascending=False
)

clustering_df.head(20)


Plain-language meaning:

Users with high clustering scores are surrounded by friends who also know each other.

Users with low clustering scores may connect people from different parts of the network.

---

## 16. Create Link Prediction Features in BigQuery

Now let’s create a real link prediction example.

The question is:

Which users are not currently friends, but look likely to be connected based on mutual friends?

The full Facebook dataset has 4,039 users. All possible user pairs would be more than 8 million pairs, which BigQuery can handle, but for a beginner tutorial we can create a smaller sample to keep it fast and cheaper.

Let’s use the first 500 users.


In [ ]:
# Create link prediction features for a smaller sample.
# This keeps the tutorial easier and less expensive to run.

feature_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.facebook_graph_demo.LinkPredictionFeatures_500` AS

WITH sampled_users AS (
  SELECT user_id
  FROM `{users_table}`
  WHERE user_id < 500
),

pairs AS (
  SELECT
    a.user_id AS user1_id,
    b.user_id AS user2_id
  FROM sampled_users a
  JOIN sampled_users b
    ON a.user_id < b.user_id
),

existing_edges AS (
  SELECT
    LEAST(source_id, target_id) AS user1_id,
    GREATEST(source_id, target_id) AS user2_id
  FROM `{edges_table}`
),

neighbors AS (
  SELECT source_id AS user_id, target_id AS friend_id
  FROM `{edges_table}`

  UNION ALL

  SELECT target_id AS user_id, source_id AS friend_id
  FROM `{edges_table}`
),

degrees AS (
  SELECT
    user_id,
    COUNT(DISTINCT friend_id) AS degree
  FROM neighbors
  GROUP BY user_id
),

mutuals AS (
  SELECT
    p.user1_id,
    p.user2_id,
    COUNT(DISTINCT n1.friend_id) AS mutual_friends
  FROM pairs p
  LEFT JOIN neighbors n1
    ON p.user1_id = n1.user_id
  LEFT JOIN neighbors n2
    ON p.user2_id = n2.user_id
   AND n1.friend_id = n2.friend_id
  WHERE n2.friend_id IS NOT NULL
  GROUP BY p.user1_id, p.user2_id
)

SELECT
  p.user1_id,
  p.user2_id,

  COALESCE(m.mutual_friends, 0) AS mutual_friends,

  COALESCE(d1.degree, 0) * COALESCE(d2.degree, 0) AS preferential_attachment,

  SAFE_DIVIDE(
    COALESCE(m.mutual_friends, 0),
    COALESCE(d1.degree, 0) + COALESCE(d2.degree, 0) - COALESCE(m.mutual_friends, 0)
  ) AS jaccard_score,

  IF(e.user1_id IS NOT NULL, 1, 0) AS label

FROM pairs p
LEFT JOIN mutuals m
  ON p.user1_id = m.user1_id
 AND p.user2_id = m.user2_id
LEFT JOIN degrees d1
  ON p.user1_id = d1.user_id
LEFT JOIN degrees d2
  ON p.user2_id = d2.user_id
LEFT JOIN existing_edges e
  ON p.user1_id = e.user1_id
 AND p.user2_id = e.user2_id;
"""

client.query(feature_sql).result()

print("Link prediction feature table created.")


Preview the feature table:


In [ ]:
features_preview_sql = f"""
SELECT *
FROM `{PROJECT_ID}.facebook_graph_demo.LinkPredictionFeatures_500`
ORDER BY mutual_friends DESC, jaccard_score DESC
LIMIT 20;
"""

client.query(features_preview_sql).to_dataframe()


The columns mean:

```text
user1_id: first user
user2_id: second user
mutual_friends: how many friends they share
preferential_attachment: whether both users are highly connected
jaccard_score: how much their friend groups overlap
label: 1 if already friends, 0 if not friends
```

---

## 17. Train a BigQuery ML Link Prediction Model

Now we train a simple logistic regression model in BigQuery ML.

The model learns patterns that separate existing friendships from non-friendships.


In [ ]:
train_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.facebook_graph_demo.facebook_link_model`
OPTIONS(
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['label'],
  data_split_method = 'AUTO_SPLIT'
) AS

SELECT
  mutual_friends,
  preferential_attachment,
  jaccard_score,
  label
FROM `{PROJECT_ID}.facebook_graph_demo.LinkPredictionFeatures_500`;
"""

client.query(train_model_sql).result()

print("BigQuery ML model trained.")


BigQuery ML supports creating and training models with SQL using `CREATE MODEL`; Google’s documentation includes logistic regression examples using `CREATE MODEL`, `ML.EVALUATE`, and `ML.PREDICT`.

---

## 18. Evaluate the Model


In [ ]:
evaluate_sql = f"""
SELECT *
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.facebook_graph_demo.facebook_link_model`,
  (
    SELECT
      mutual_friends,
      preferential_attachment,
      jaccard_score,
      label
    FROM `{PROJECT_ID}.facebook_graph_demo.LinkPredictionFeatures_500`
  )
);
"""

client.query(evaluate_sql).to_dataframe()


For a real article, be careful with the interpretation.

This model is useful for learning, but it is not a production recommendation system. A better real evaluation would use time: train on earlier friendships and test whether the model predicts later friendships.

---

## 19. Predict Possible Missing Friendships

Now score pairs that are not currently connected.


In [ ]:
predict_sql = f"""
WITH predictions AS (
  SELECT *
  FROM ML.PREDICT(
    MODEL `{PROJECT_ID}.facebook_graph_demo.facebook_link_model`,
    (
      SELECT
        user1_id,
        user2_id,
        mutual_friends,
        preferential_attachment,
        jaccard_score
      FROM `{PROJECT_ID}.facebook_graph_demo.LinkPredictionFeatures_500`
      WHERE label = 0
    )
  )
)

SELECT
  user1_id,
  user2_id,
  mutual_friends,
  preferential_attachment,
  ROUND(jaccard_score, 4) AS jaccard_score,
  prob.prob AS predicted_friendship_probability
FROM predictions
CROSS JOIN UNNEST(predicted_label_probs) AS prob
WHERE prob.label = 1
ORDER BY predicted_friendship_probability DESC
LIMIT 20;
"""

recommendations_df = client.query(predict_sql).to_dataframe()

recommendations_df


Plain-language meaning:

These pairs are not currently friends in the dataset, but the model thinks they look similar to existing friendship pairs.

This is link prediction.

---

## 20. Visualize One Predicted Link


In [ ]:
# Pick the highest-scoring predicted friendship.
top_prediction = recommendations_df.iloc[0]

user_a = int(top_prediction["user1_id"])
user_b = int(top_prediction["user2_id"])

print("Top predicted link:", user_a, user_b)


Now build a small local graph around those two users.


In [ ]:
# Get neighbors around both users.
local_edges_sql = f"""
WITH local_users AS (
  SELECT {user_a} AS user_id
  UNION ALL
  SELECT {user_b} AS user_id
),

neighbors AS (
  SELECT source_id, target_id
  FROM `{edges_table}`
  WHERE source_id IN (SELECT user_id FROM local_users)
     OR target_id IN (SELECT user_id FROM local_users)
)

SELECT *
FROM neighbors
LIMIT 200;
"""

local_edges_df = client.query(local_edges_sql).to_dataframe()

# Build local graph.
G_local = nx.from_pandas_edgelist(
    local_edges_df,
    source="source_id",
    target="target_id",
    create_using=nx.Graph()
)

# Add the predicted link as a dashed visual connection.
G_future = G_local.copy()
G_future.add_edge(user_a, user_b)

# Draw.
pos = nx.spring_layout(G_local, seed=42)

plt.figure(figsize=(10, 8))

nx.draw_networkx_nodes(G_local, pos, node_size=300)
nx.draw_networkx_edges(G_local, pos, width=1)
nx.draw_networkx_labels(G_local, pos, font_size=7)

# Draw predicted edge.
nx.draw_networkx_edges(
    G_future,
    pos,
    edgelist=[(user_a, user_b)],
    width=3,
    style="dashed"
)

plt.title(f"Predicted Possible Friendship: User {user_a} and User {user_b}")
plt.axis("off")
plt.show()


---

## How to Explain This in the Article

You can write:

Instead of using a made-up friendship map, we used a real anonymized Facebook friendship network from Stanford SNAP. Each anonymized user became a node, and each friendship became an edge. We loaded the data into BigQuery, created a property graph, analyzed degree and centrality, visualized part of the network in Colab, and then built a simple link prediction model using BigQuery ML.

The model used graph-based clues such as mutual friends, Jaccard similarity, and preferential attachment. In plain language, it asked:

Do these two users share many friends?

Do their friend groups overlap?

Are both users already well-connected?

Based on those clues, the model scored possible missing friendship links.

This turns the beginner concept of “people connected by friendships” into a real graph analytics workflow.
